# 🛡️ Enterprise Data Governance for Data Engineering

This notebook demonstrates the end-to-end implementation of an **Enterprise Data Governance Framework**. Data governance ensures that data assets are secure, cataloged, classified, compliant, and tracked across their entire lifecycle.

### Key Governance Capabilities Covered:
1. 📖 **Data Cataloging & Ingestion**: Registering and searching table & column-level metadata.
2. 🏷️ **Data Classification & PII Scanning**: Automatically detecting PII and tagging sensitivity levels.
3. 🔗 **Data Lineage & Impact Analysis**: Recording relationships, detecting dependency cycles, and assessing the impact of schema changes.
4. 🔐 **Role-Based Access Control (RBAC) & Masking**: Defining user access roles and dynamically redacting or hashing sensitive columns.
5. 📋 **Compliance & Retention Lifecycle**: Supporting GDPR workflows (Subject Access Request, Right to Be Forgotten) and enforcing retention timelines.


In [ ]:
import os
import sys
import logging
import pandas as pd

# Add src to python path to import modules
sys.path.append(os.path.abspath("../"))

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")

# Check imports
from src.catalog.data_catalog import DataCatalog, CatalogEntry
from src.catalog.metadata_manager import MetadataManager
from src.catalog.glossary_manager import GlossaryManager, GlossaryTerm
from src.classification.pii_detector import PIIDetector
from src.classification.data_classifier import DataClassifier
from src.classification.sensitivity_tagger import SensitivityTagger
from src.lineage.lineage_tracker import LineageTracker
from src.lineage.lineage_graph import LineageGraph
from src.lineage.lineage_reporter import LineageReporter
from src.access_control.rbac_manager import RBACManager
from src.access_control.policy_engine import PolicyEngine
from src.access_control.audit_logger import AuditLogger
from src.compliance.gdpr_compliance import GDPRComplianceChecker
from src.compliance.data_retention import DataRetentionManager
from src.compliance.compliance_reporter import ComplianceReporter

print("Environment setup complete! All modules imported successfully.")


## 📖 1. Data Cataloging & Metadata Ingestion

The **Data Catalog** is the foundation of governance, allowing data discovery and capturing physical properties of our data platform. We will initialize a local catalog database and ingest standard schemas from our CSV asset inventory.


In [ ]:
# Initialize catalog and metadata manager
db_file = "governance_notebook.db"
if os.path.exists(db_file):
    os.remove(db_file)

catalog = DataCatalog(db_path=db_file)
metadata_mgr = MetadataManager(catalog)
glossary_mgr = GlossaryManager(db_path=db_file)

# Bulk ingest metadata from CSV
csv_path = "../sample-data/data_assets_inventory.csv"
num_ingested = metadata_mgr.ingest_from_csv(csv_path)
print(f"Ingested {num_ingested} column-level assets into Data Catalog.")

# Define glossary term
term = GlossaryTerm(
    term_name="SSN",
    definition="Social Security Number, a 9-digit identifier issued to US citizens and residents.",
    category="Customer",
    owner="Security Compliance Team",
    sensitivity_level="RESTRICTED"
)
glossary_mgr.create_term(term)
print("Created Glossary Term: SSN")

# Query assets from catalog
search_results = catalog.search_assets("email")
for r in search_results:
    print(f"Search Result: {r['database_name']}.{r['schema_name']}.{r['table_name']}.{r['column_name']} ({r['data_type']}) - Owner: {r['owner']}")


## 🏷️ 2. Data Classification & PII Scanning

Automated PII detection saves manual work. We will run our automated PII scanning engine on a sample dataframe to identify columns containing emails, SSNs, credit cards, and phone numbers, and apply sensitivity tags.


In [ ]:
# Load sample PII data
df = pd.read_csv("../sample-data/sample_pii_data.csv")
print("Sample DataFrame:")
print(df.head(2))

# Initialize DataClassifier with the config file path
classifier = DataClassifier(catalog, config_path="../configs/governance_config.yaml")

# Run scanning
classifications = classifier.classify_and_tag_dataframe(
    df=df,
    database_name="prod_dw",
    schema_name="customers",
    table_name="users",
    trigger_by="scheduled_crawler"
)

print("\nScanning Completed! Column classifications applied:")
for col, cls in classifications.items():
    print(f" - Column: {col} -> Sensitivity Level: {cls}")


## 🔗 3. Data Lineage & Dependency Tracking

Tracking dependencies between tables and columns allows us to assess downstream impact prior to schema upgrades and debug upstream data quality anomalies. We will record lineage edges, traverse dependencies, and export a Mermaid markdown layout diagram.


In [ ]:
# Create LineageTracker and record operations
tracker = LineageTracker()
tracker.record_lineage("raw_orders", "bronze_orders", "COPY", pipeline_name="ingest_pipeline")
tracker.record_lineage("bronze_orders", "silver_orders", "CLEAN", pipeline_name="clean_pipeline")
tracker.record_lineage("silver_orders", "gold_orders_summary", "AGGREGATE", pipeline_name="agg_pipeline")
tracker.record_lineage("silver_customers", "gold_orders_summary", "JOIN", pipeline_name="agg_pipeline")

# Build DAG graph
graph = LineageGraph()
graph.build_from_tracker(tracker)

# Traverse Graph
upstream = graph.get_upstream_dependencies("gold_orders_summary")
downstream = graph.get_downstream_impact("raw_orders")
roots = graph.find_root_causes("gold_orders_summary")

print(f"Upstream Dependencies of 'gold_orders_summary': {upstream}")
print(f"Downstream Impact of 'raw_orders': {downstream}")
print(f"Root Cause Sources of 'gold_orders_summary': {roots}")

# Generate Mermaid layout
reporter = LineageReporter(graph)
print("\nGenerated Mermaid.js Diagram:")
print(reporter.generate_mermaid_diagram())


## 🔐 4. Access Control & Dynamic Data Masking

Sensitive data should only be visible to authorized roles. We will define an access policy and dynamically mask raw column values (partial redaction, hashing) based on user roles and sensitivity levels.


In [ ]:
# Initialize RBAC and Policy Engine
rbac = RBACManager()
engine = PolicyEngine(rbac)
audit = AuditLogger()

# Register users with different roles
rbac.register_user("user_analyst", "analyst_john", roles={"DATA_ANALYST"})
rbac.register_user("user_admin", "admin_sarah", roles={"DATA_ADMIN"})

# Raw PII values
email = "john.doe@company.com"
ssn = "123-45-6789"

# Evaluate Analyst access
print("--- Analyst Access ---")
email_res = engine.evaluate_field_access("user_analyst", email, pii_type="EMAIL", sensitivity_level="RESTRICTED")
ssn_res = engine.evaluate_field_access("user_analyst", ssn, pii_type="SSN", sensitivity_level="RESTRICTED")
print(f"Email Decision: {email_res['decision']} -> Value: {email_res['value']}")
print(f"SSN Decision: {ssn_res['decision']} -> Value: {ssn_res['value']}")
audit.log_access("user_analyst", "READ", "email", email_res["decision"], {"unmasked": not email_res["masked"]})

# Evaluate Admin access
print("\n--- Admin Access ---")
email_res_admin = engine.evaluate_field_access("user_admin", email, pii_type="EMAIL", sensitivity_level="RESTRICTED")
ssn_res_admin = engine.evaluate_field_access("user_admin", ssn, pii_type="SSN", sensitivity_level="RESTRICTED")
print(f"Email Decision: {email_res_admin['decision']} -> Value: {email_res_admin['value']}")
print(f"SSN Decision: {ssn_res_admin['decision']} -> Value: {ssn_res_admin['value']}")
audit.log_access("user_admin", "READ", "email", email_res_admin["decision"], {"unmasked": not email_res_admin["masked"]})


## 📋 5. GDPR Compliance & Retention Lifecycle

We will process regulatory tasks such as a Subject Access Request (SAR), execute Right to Be Forgotten (RTBF) record deletion, and verify data lifecycle ages against corporate retention policies.


In [ ]:
# Initialize GDPR Compliance Checker
checker = GDPRComplianceChecker(catalog)
ret_mgr = DataRetentionManager()

# Mock dataset for GDPR operations
gdpr_datasets = {
    "raw_customers": [
        {"customer_id": "CUST_1001", "name": "Alice Smith", "email": "alice@smith.com"},
        {"customer_id": "CUST_1002", "name": "Bob Jones", "email": "bob@jones.com"}
    ]
}

# 1. Subject Access Request (SAR)
sar_report = checker.process_subject_access_request("CUST_1001", gdpr_datasets)
print("--- Subject Access Request ---")
print(f"Subject: {sar_report.subject_id} | Total Records Found: {sar_report.total_records_found}")
print(sar_report.retrieved_records)

# 2. Right to Be Forgotten (RTBF) - Anonymization
erasure_report = checker.process_right_to_be_forgotten("CUST_1001", gdpr_datasets, anonymize_fields=["name", "email"])
print("\n--- Right to Be Forgotten ---")
print(f"Anonymized Records: {erasure_report.records_anonymized_count}")
print("Remaining dataset rows:")
print(gdpr_datasets["raw_customers"])

# 3. Data Retention lifecycle
# Ingest catalog entries to evaluate in reporter
catalog.register_asset(CatalogEntry("users_table", "users", "customer_360", "identity@co.com", "table", "RESTRICTED"))
catalog.register_asset(CatalogEntry("mktg_table", "campaigns", "marketing", "mktg@co.com", "table", "PUBLIC", created_at="2015-01-01T00:00:00Z"))

# Generate Compliance Posture Summary
reporter = ComplianceReporter(catalog, audit, ret_mgr)
summary = reporter.generate_summary_report()
print("\n--- Enterprise Compliance Posture Summary ---")
print(f"Compliance Score: {summary.compliance_score}/100.0 | Risk Level: {summary.risk_level}")
print(f"Total cataloged assets: {summary.total_assets}")
print("Recommendations for compliance fixing:")
for rec in summary.recommendations:
    print(f" * {rec}")
